# Regression Machine Test — Simple Exam Reference

**Copy/paste reference.**

This version intentionally keeps the workflow simple and exam-friendly.

### Included
- Basic EDA
- Missing values
- Duplicates
- Histograms
- Count plots
- Bar plots
- Box plots
- Scatter plots
- Regression plots
- Correlation heatmap
- IQR outlier detection
- Manual missing-value treatment
- Manual categorical encoding
- Manual train/test split
- StandardScaler
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting Regressor
- MAE, RMSE, R²
- Actual vs Predicted
- Residual plot
- GridSearchCV
- Feature importance
- Pickle/joblib saving
- Streamlit + Flask reference

### NOT included
- SimpleImputer
- ColumnTransformer
- Pipeline
- automatic feature detection
- violin plot
- pair plot


In [ ]:
# 1. IMPORT LIBRARIES

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

sns.set_theme(style="whitegrid")

FILE_PATH = "/content/your_dataset.csv"
df = pd.read_csv(FILE_PATH)

print(df.shape)
display(df.head())

## 2. BASIC UNDERSTANDING

In [ ]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nInfo:")
df.info()

print("\nStatistics:")
display(df.describe().T)

## 3. DUPLICATES

In [ ]:
print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().reset_index(drop=True)

print("Shape after removing duplicates:", df.shape)

## 4. MISSING VALUES

In [ ]:
print(df.isnull().sum())

## 5. MISSING VALUES — VISUALIZATION

In [ ]:
missing = df.isnull().sum()

plt.figure(figsize=(10, 5))
sns.barplot(x=missing.index, y=missing.values)
plt.xticks(rotation=90)
plt.xlabel("Columns")
plt.ylabel("Missing Values")
plt.title("Missing Values by Column")
plt.tight_layout()
plt.show()

## 6. TARGET DISTRIBUTION

In [ ]:
TARGET = "Target"

plt.figure(figsize=(8, 5))
sns.histplot(df[TARGET].dropna(), bins=30, kde=True)
plt.xlabel(TARGET)
plt.ylabel("Frequency")
plt.title(f"Distribution of {TARGET}")
plt.tight_layout()
plt.show()

## 7. TARGET BOXPLOT

In [ ]:
plt.figure(figsize=(8, 4))
sns.boxplot(x=df[TARGET])
plt.xlabel(TARGET)
plt.title(f"Box Plot of {TARGET}")
plt.tight_layout()
plt.show()

## 8. HISTOGRAM — NUMERICAL COLUMN

In [ ]:
plt.figure(figsize=(8, 5))
sns.histplot(data=df, x="Numerical_Column", bins=30, kde=True)
plt.xlabel("Numerical_Column")
plt.ylabel("Frequency")
plt.title("Distribution of Numerical_Column")
plt.tight_layout()
plt.show()

## 9. BOX PLOT — OUTLIERS

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Numerical_Column")
plt.xlabel("Numerical_Column")
plt.title("Box Plot of Numerical_Column")
plt.tight_layout()
plt.show()

## 10. COUNT PLOT — CATEGORICAL COLUMN

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x="Categorical_Column")
plt.xlabel("Categorical_Column")
plt.ylabel("Count")
plt.title("Count Plot of Categorical_Column")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 11. BAR PLOT — CATEGORY VS AVERAGE TARGET

In [ ]:
summary = df.groupby("Categorical_Column")[TARGET].mean().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=summary.index, y=summary.values)
plt.xlabel("Categorical_Column")
plt.ylabel(f"Average {TARGET}")
plt.title(f"Average {TARGET} by Categorical_Column")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 12. BOX PLOT — CATEGORY VS TARGET

In [ ]:
plt.figure(figsize=(9, 5))
sns.boxplot(data=df, x="Categorical_Column", y=TARGET)
plt.xlabel("Categorical_Column")
plt.ylabel(TARGET)
plt.title(f"{TARGET} by Categorical_Column")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 13. SCATTER PLOT — FEATURE VS TARGET

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="Numerical_Column", y=TARGET)
plt.xlabel("Numerical_Column")
plt.ylabel(TARGET)
plt.title(f"Numerical_Column vs {TARGET}")
plt.tight_layout()
plt.show()

## 14. REGRESSION PLOT — FEATURE VS TARGET

In [ ]:
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x="Numerical_Column", y=TARGET)
plt.xlabel("Numerical_Column")
plt.ylabel(TARGET)
plt.title(f"Regression Plot: Numerical_Column vs {TARGET}")
plt.tight_layout()
plt.show()

## 15. CORRELATION HEATMAP

In [ ]:
numeric_cols = [
    "Numerical_Column_1",
    "Numerical_Column_2",
    "Numerical_Column_3",
    TARGET
]

plt.figure(figsize=(10, 7))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

## 16. TARGET CORRELATION

In [ ]:
correlation = df[numeric_cols].corr()[TARGET].drop(TARGET)
print(correlation.sort_values(key=abs, ascending=False))

## 17. IQR OUTLIER DETECTION

In [ ]:
col = "Numerical_Column"

Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

outliers = df[(df[col] < lower) | (df[col] > upper)]

print("Q1:", Q1)
print("Q3:", Q3)
print("IQR:", IQR)
print("Lower limit:", lower)
print("Upper limit:", upper)
print("Number of outliers:", len(outliers))

## 18. OUTLIER TREATMENT — CAPPING

In [ ]:
# Use only if capping is suitable for the problem.

df[col] = df[col].clip(lower=lower, upper=upper)

print("Outliers capped.")

## 19. MISSING VALUE TREATMENT — MANUAL

In [ ]:
# Numeric column -> median
df["Numerical_Column"] = df["Numerical_Column"].fillna(
    df["Numerical_Column"].median()
)

# Categorical column -> mode
df["Categorical_Column"] = df["Categorical_Column"].fillna(
    df["Categorical_Column"].mode()[0]
)

print(df.isnull().sum())

## 20. CATEGORICAL ENCODING — LABEL ENCODING

In [ ]:
# Use when the categorical variable has two meaningful classes.
# Example: Yes/No

df["Binary_Column"] = df["Binary_Column"].map({
    "No": 0,
    "Yes": 1
})

print(df["Binary_Column"].value_counts())

## 21. ONE-HOT ENCODING

In [ ]:
# Use for nominal categorical variables with multiple categories.

df = pd.get_dummies(
    df,
    columns=["Categorical_Column"],
    drop_first=True
)

display(df.head())

## 22. FEATURE ENGINEERING — COMMON PATTERNS

In [ ]:
# Examples — change column names/formulas according to the dataset.

# df["Total_Value"] = df["Quantity"] * df["Unit_Price"]

# df["Price_Per_Unit"] = df["Total_Price"] / df["Area"]

# df["Age"] = df["Current_Year"] - df["Birth_Year"]

# df["Total_Experience"] = df["Experience_Years"] + df["Training_Years"]

display(df.head())

## 23. DEFINE X AND y

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

print("X shape:", X.shape)
print("y shape:", y.shape)

## 24. TRAIN TEST SPLIT

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 25. STANDARD SCALING

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

## 26. MODEL 1 — LINEAR REGRESSION

In [ ]:
model_lr = LinearRegression()

model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)

print("MAE :", mean_absolute_error(y_test, y_pred_lr))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_lr)))
print("R²  :", r2_score(y_test, y_pred_lr))

## 27. MODEL 2 — RIDGE REGRESSION

In [ ]:
model_ridge = Ridge(alpha=1.0)

model_ridge.fit(X_train_scaled, y_train)

y_pred_ridge = model_ridge.predict(X_test_scaled)

print("MAE :", mean_absolute_error(y_test, y_pred_ridge))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_ridge)))
print("R²  :", r2_score(y_test, y_pred_ridge))

## 28. MODEL 3 — DECISION TREE REGRESSOR

In [ ]:
model_dt = DecisionTreeRegressor(
    max_depth=8,
    random_state=42
)

model_dt.fit(X_train, y_train)

y_pred_dt = model_dt.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_dt))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_dt)))
print("R²  :", r2_score(y_test, y_pred_dt))

## 29. MODEL 4 — RANDOM FOREST REGRESSOR

In [ ]:
model_rf = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_rf))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_rf)))
print("R²  :", r2_score(y_test, y_pred_rf))

## 30. MODEL 5 — GRADIENT BOOSTING

In [ ]:
model_gb = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

model_gb.fit(X_train, y_train)

y_pred_gb = model_gb.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_gb))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_gb)))
print("R²  :", r2_score(y_test, y_pred_gb))

## 31. MODEL COMPARISON

In [ ]:
results = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Ridge Regression",
        "Decision Tree",
        "Random Forest",
        "Gradient Boosting"
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_dt),
        mean_absolute_error(y_test, y_pred_rf),
        mean_absolute_error(y_test, y_pred_gb)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y_test, y_pred_lr)),
        np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
        np.sqrt(mean_squared_error(y_test, y_pred_dt)),
        np.sqrt(mean_squared_error(y_test, y_pred_rf)),
        np.sqrt(mean_squared_error(y_test, y_pred_gb))
    ],
    "R2": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_dt),
        r2_score(y_test, y_pred_rf),
        r2_score(y_test, y_pred_gb)
    ]
})

display(results.sort_values("RMSE"))

## 32. ACTUAL vs PREDICTED

In [ ]:
# Choose the final model
y_pred = y_pred_rf

plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.show()

## 33. RESIDUAL DISTRIBUTION

In [ ]:
residuals = y_test - y_pred

plt.figure(figsize=(8, 5))
sns.histplot(residuals, bins=30, kde=True)
plt.xlabel("Residual")
plt.title("Residual Distribution")
plt.tight_layout()
plt.show()

## 34. RESIDUALS vs PREDICTED

In [ ]:
plt.figure(figsize=(8, 5))
sns.scatterplot(x=y_pred, y=residuals)
plt.axhline(0, linestyle="--")
plt.xlabel("Predicted Values")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted Values")
plt.tight_layout()
plt.show()

## 35. GRIDSEARCHCV — RANDOM FOREST

In [ ]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    RandomForestRegressor(
        random_state=42,
        n_jobs=-1
    ),
    param_grid=param_grid,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best Parameters:")
print(grid.best_params_)

print("Best CV RMSE:")
print(-grid.best_score_)

## 36. TUNED MODEL EVALUATION

In [ ]:
best_model = grid.best_estimator_

y_pred_best = best_model.predict(X_test)

print("MAE :", mean_absolute_error(y_test, y_pred_best))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred_best)))
print("R²  :", r2_score(y_test, y_pred_best))

## 37. TUNED MODEL — ACTUAL vs PREDICTED

In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(x=y_test, y=y_pred_best)

min_value = min(y_test.min(), y_pred_best.min())
max_value = max(y_test.max(), y_pred_best.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--"
)

plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")
plt.title("Tuned Model: Actual vs Predicted")
plt.tight_layout()
plt.show()

## 38. FEATURE IMPORTANCE — RANDOM FOREST

In [ ]:
importance = pd.Series(
    best_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

display(importance.head(20))

plt.figure(figsize=(10, 7))
sns.barplot(
    x=importance.head(20).values,
    y=importance.head(20).index
)
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances")
plt.tight_layout()
plt.show()

## 39. SAVE MODEL + SCALER

In [ ]:
joblib.dump(best_model, "regression_model.pkl")
joblib.dump(scaler, "scaler.pkl")

print("Saved:")
print("regression_model.pkl")
print("scaler.pkl")

## 40. LOAD MODEL

In [ ]:
loaded_model = joblib.load("regression_model.pkl")

# Example:
# prediction = loaded_model.predict(new_data)
# print("Prediction:", prediction[0])

# 41. STREAMLIT — EXAM REFERENCE

Save as `app.py`.

```python
import streamlit as st
import pandas as pd
import joblib

model = joblib.load("regression_model.pkl")

st.title("Regression Prediction")

# Change these inputs according to the dataset
feature1 = st.number_input("Feature 1")
feature2 = st.number_input("Feature 2")

if st.button("Predict"):
    input_data = pd.DataFrame([{
        "Feature1": feature1,
        "Feature2": feature2
    }])

    prediction = model.predict(input_data)

    st.success(f"Predicted value: {prediction[0]:.2f}")
```

# 42. FLASK — EXAM REFERENCE

Save as `app.py`.

```python
from flask import Flask, request, jsonify
import pandas as pd
import joblib

app = Flask(__name__)

model = joblib.load("regression_model.pkl")

@app.route("/")
def home():
    return "Regression API is running"

@app.route("/predict", methods=["POST"])
def predict():
    data = request.get_json()

    input_data = pd.DataFrame([data])

    prediction = model.predict(input_data)

    return jsonify({
        "prediction": float(prediction[0])
    })

if __name__ == "__main__":
    app.run(debug=True)
```

# ⭐ FINAL REGRESSION EXAM WORKFLOW

### 1. Load
```python
df = pd.read_csv("file.csv")
```

### 2. Understand
```python
df.shape
df.head()
df.info()
df.describe()
```

### 3. Clean
```python
df.isnull().sum()
df.duplicated().sum()
df.drop_duplicates()
```

### 4. EDA
```python
sns.countplot(...)
sns.histplot(...)
sns.boxplot(...)
sns.barplot(...)
sns.scatterplot(...)
sns.regplot(...)
sns.heatmap(...)
```

### 5. Treat missing values
- Numerical → median
- Categorical → mode

### 6. Outliers
- Detect with IQR
- Decide whether to remove/cap/keep based on the problem

### 7. Encode
- Binary categorical → mapping
- Multi-category nominal → `pd.get_dummies()`

### 8. Feature engineering
Create useful domain-based features.

### 9. Split
```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
```

### 10. Scale
Use `StandardScaler` mainly for models such as Linear/Ridge Regression.

### 11. Train at least 3 models
- Linear Regression
- Decision Tree Regressor
- Random Forest Regressor
- Gradient Boosting is another option

### 12. Evaluate
```python
MAE
RMSE
R²
```

### 13. Tune
Use `GridSearchCV`.

### 14. Analyze
- Actual vs Predicted
- Residual distribution
- Residuals vs predicted
- Feature importance

### 15. Save
```python
joblib.dump(model, "regression_model.pkl")
```

### 16. Deploy
Use the framework requested in the machine test: Streamlit or Flask.
